# Notebook 5: Capstone

Now you build a small agent end to end. The scenario is a **field data helper**
that answers questions from a small program dataset. Most of it is built. Your
job is to finish the marked TODOs and then try your own questions.

Other scenario ideas you can adapt this to: a multilingual FAQ assistant, a
report-drafting helper, or a simple indicator calculator. Pick one that fits
your work.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-opus-4-8 is the most capable. For faster, cheaper
#    runs while practising you can switch this to "claude-haiku-4-5".
MODEL = "claude-opus-4-8"
print("Using model:", MODEL)

## The dataset and the first tool

A lookup tool is provided and works.

In [ ]:
# A tiny, made-up dataset for the workshop: a few program indicators by region.
# In a real project this might come from a spreadsheet or a database.
program_data = {
    "north": {"households_reached": 1200, "clinics": 4, "literacy_rate": 0.71},
    "south": {"households_reached": 900, "clinics": 3, "literacy_rate": 0.63},
    "east": {"households_reached": 1500, "clinics": 6, "literacy_rate": 0.78},
}

lookup_indicator_tool = {
    "name": "lookup_indicator",
    "description": "Look up one indicator for one region. Regions: north, south, east. Indicators: households_reached, clinics, literacy_rate.",
    "input_schema": {
        "type": "object",
        "properties": {
            "region": {"type": "string", "description": "north, south, or east"},
            "indicator": {"type": "string", "description": "households_reached, clinics, or literacy_rate"},
        },
        "required": ["region", "indicator"],
    },
}


def run_lookup_indicator(region, indicator):
    region = region.lower().strip()
    indicator = indicator.lower().strip()
    if region not in program_data:
        return "Error: unknown region. Use north, south, or east."
    if indicator not in program_data[region]:
        return "Error: unknown indicator. Use households_reached, clinics, or literacy_rate."
    return str(program_data[region][indicator])

## A calculator tool (so the agent can compute, for example averages)

In [ ]:
import ast
import operator

# A tool is described by a name, a plain description, and the inputs it needs.
# The model reads the description to decide when to use the tool.
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the result. Use this whenever the user asks for a calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "An arithmetic expression, for example '12 * 37'.",
            }
        },
        "required": ["expression"],
    },
}

# The safe operations our calculator allows: plus, minus, times, divide, and
# negative numbers. Nothing else is permitted.
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    # Walk the parsed expression and only allow numbers and the safe operations
    # above. Anything else (names, function calls) raises an error, so no
    # arbitrary code can run. This is the safe alternative to eval().
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def run_calculator(expression):
    # Beginners can treat this as a black box: give it an expression, get the
    # result. It parses the text and evaluates only safe arithmetic, so it
    # cannot run arbitrary code.
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree))
    except Exception:
        return "Error: I could only handle a simple arithmetic expression like '12 * 37'."

## Your tasks

1. Register both tools in the `tools` list and the `run_tool` dispatcher below.
2. Run the agent and ask it a question that needs a lookup and a calculation,
   for example: "What is the total number of clinics across all three regions?"
3. Stretch goal: add one more tool of your own (for example one that returns the
   region with the most households).

The instructor notebook shows one complete solution.

In [ ]:
# TODO: include both tools.
tools = [lookup_indicator_tool]  # add calculator_tool

def run_tool(name, tool_input):
    if name == "lookup_indicator":
        return run_lookup_indicator(tool_input["region"], tool_input["indicator"])
    # TODO: add a branch for "calculator" that calls run_calculator.
    return "Error: unknown tool."

SYSTEM = (
    "You are a field data helper. Use the tools to look up indicators and to do "
    "any arithmetic, one step at a time, then give a short, clear answer."
)

## The agent loop

The same reusable loop from earlier notebooks.

In [ ]:
def run_agent(question, max_steps=8):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=tools, messages=messages,
        )
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print("Thought:", block.text.strip())
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    return block.text
            return ""
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print("Action:", block.name, block.input)
                observation = run_tool(block.name, block.input)
                print("Observation:", observation)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": observation})
        messages.append({"role": "user", "content": tool_results})
    return "Stopped after the maximum number of steps."

## Try it

In [ ]:
question = "What is the total number of clinics across all three regions?"
print(run_agent(question))

## Wrap up

You built an agent that plans, looks up real data, computes, and answers. That
is the same loop behind much larger agents. Well done.

When you are ready, write a short note on which tools your agent uses and how it
plans. That note plus this notebook is your capstone submission.